# Fase 2 — Estandarización y anonimización de participaciones estudiantiles

**Trabajo de Grado:** Sistema de contraste de Redes Bayesianas y LSTM para la predicción explicable de la participación en el aula de los estudiantes de la Universidad Metropolitana
**Autores:** Nelson Carrillo, Luis Pérez — **Tutor:** Fernando Torre Mora — **Co-tutor:** José Alberto Peña E.

Este notebook orquesta la **Fase 2 (Sprint 1)** de la metodología del anteproyecto: *"levantamiento, limpieza y estandarización de datos"*. La lógica de extracción vive en módulos `.py` separados en esta misma carpeta (ver `README.md`); aquí solo se importan, se ejecutan y se revisan los resultados.

| Módulo | Responsabilidad |
|---|---|
| `cronogramas.py` | Extrae la tabla semana → tema de cada cronograma (`.docx` / `.xlsx`) |
| `limpieza.py` | Normalización de cédulas, nombres, fechas y códigos de asistencia |
| `extractores.py` | Un extractor por cada uno de los 3 formatos de archivo fuente |
| `fuentes.py` | Catálogo declarativo de las 11 fuentes y su despacho al extractor correspondiente |
| `anonimizacion.py` | Mapa cédula → `estudiante_id`, consistente entre archivos |
| `validacion.py` | Chequeos post-procesamiento (sin datos personales) |
| `pipeline.py` | Orquesta todo lo anterior; también se puede correr como script (`python3 pipeline.py`) |

## Cómo correrlo

Abre este notebook desde `preprocesamiento/` y ejecuta todas las celdas en orden (`Run All`). No requiere argumentos.

**Salida** (en `Datos Tesis/_procesado/`):
- `*_participaciones.csv` — 13 archivos estandarizados y anonimizados (esquema en la Tabla 4 del informe).
- `log_limpieza.txt` — bitácora pública de las transformaciones (sin datos personales).
- `_confidencial/mapeo_estudiantes.csv` y `_confidencial/log_limpieza_detalle.txt` — datos personales (⚠️ excluidos de git vía `.gitignore`, nunca se suben).

## 1. Cargar los módulos del pipeline

Asume que el notebook corre con directorio de trabajo `preprocesamiento/` (el default de Jupyter/VS Code al abrirlo).

In [ ]:
from pathlib import Path
import pandas as pd

from pipeline import ejecutar, COLUMNAS_SALIDA

assert Path("pipeline.py").exists(), "Corre este notebook desde la carpeta 'preprocesamiento/'."
print("Modulos cargados OK")


## 2. Ejecutar el pipeline completo

Cronogramas → extracción de las 11 fuentes → anonimización → escritura de CSV y logs → validación.

In [ ]:
resumen = ejecutar()

print(f"{len(resumen['archivos_generados'])} archivos CSV generados en {resumen['out_dir']}")
print(f"{resumen['n_estudiantes']} estudiantes unicos anonimizados")


## 3. Resultado por archivo

In [ ]:
pd.DataFrame(resumen['archivos_generados'], columns=['fuente', 'archivo', 'filas']).assign(
    archivo=lambda d: d['archivo'].apply(lambda p: p.name)
)


## 4. Bitácora de transformaciones

Se imprime aquí mismo (no contiene datos personales — ver `README.md` para la política de qué va en el log público vs. el confidencial).

In [ ]:
print("\n".join(resumen['log_publico']))


## 5. Vista rápida de un CSV de salida

Solo para confirmar visualmente el esquema (Tabla 4). No imprime nada del mapeo de anonimización.

In [ ]:
ejemplo = pd.read_csv(resumen['out_dir'] / "algoritmos_2425-2_sec1_participaciones.csv")
assert list(ejemplo.columns) == COLUMNAS_SALIDA
ejemplo.head()
